# R10 · TwinCheck demo with live web evidence (Rathi)

**Runtime → T4 GPU**, switch on **Notebook access** for `GITHUB_TOKEN` and `HF_TOKEN` in the 🔑 panel, then **Run all**.

Steps: 1 setup → 2 tests → 3 run the examples and push them (optional) → 4 launch the app (public `gradio.live` link; the cell keeps running while the demo is open).

Settings in cell 1:
- `EVIDENCE = "web"` (default): when the evidence box is empty, each claim is searched on the web (DuckDuckGo, no API key). Fact-checking sites are removed and the reranker keeps the 5 most relevant results, each with its link. `"offline"` uses the offline library (163,244 snippets) instead. Pasted evidence is always used as it is. Live search is for the demo only; the reported experiments use the offline library.
- `VERDICT = "head"` (default): Himasri's trained twin-loss verdict head. `"rule"` uses the stand-in rule (red banner on screen).
- `FAST = True` (default): rule-based claim splitting + template notes (~5 min to the link). `FAST = False` uses Qwen3-8B for claims and notes.
- The image card (F11) works too: upload a picture, or use the **Text + image cases** buttons.

Script and shot list: `notes/R10_demo_video_script.md`.


In [ ]:
# 1. Setup
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
BRANCH = 'rathi/R10-live-web'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}

EVIDENCE = "web"   # "web" = live web search when no evidence is pasted; "offline" = offline library
VERDICT = "head"   # "head" = trained twin-loss verdict head; "rule" = stand-in (labelled on screen)
FAST = True        # True: rule claim split + template notes (~5 min); False: Qwen3-8B (slow first load)

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
!pip install -q -U gradio rank_bm25 faiss-cpu ddgs
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
os.environ['TWINCHECK_NLI'] = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'   # zero-shot NLI
os.environ['TWINCHECK_VERDICT'] = 'models/verdict_twin_s42' if VERDICT == 'head' else 'none'
if FAST:
    !python -m spacy download -q en_core_web_sm
    os.environ.update(TWINCHECK_EXTRACTOR='split', TWINCHECK_EXPLAINER='template',
                      TWINCHECK_SPACY='en_core_web_sm')
else:
    !pip install -q -U bitsandbytes
    !python -m spacy download -q en_core_web_trf
    for k in ('TWINCHECK_EXTRACTOR', 'TWINCHECK_EXPLAINER', 'TWINCHECK_SPACY'):
        os.environ.pop(k, None)
# image detector for the F11 image card (Git LFS, 343 MB)
CKPT = 'models/F5/vit_b16_s42.pt'
!which git-lfs || apt-get -qq install -y git-lfs
!git lfs install
!git lfs pull --include {CKPT}
if os.path.getsize(CKPT) < 1_000_000:
    !wget -q -O {CKPT} https://media.githubusercontent.com/media/himaallu/Group-Project/{BRANCH}/{CKPT}
print(f'image detector: {os.path.getsize(CKPT) / 1e6:.0f} MB')
# offline evidence library (~330 MB), used when EVIDENCE = "offline"
!python -m src.retrieval.build_index download --repo RathiAbeth/twincheck-index-v1
!python -m src.retrieval.live "Gold rates in Dubai 24K AED 437.75" --k 2
!python -m src.pipeline --status


## 2. Tests

In [ ]:
!python -m pytest tests/demo tests/explain tests/retrieval/test_live.py tests/test_pipeline.py -q

## 3. Run the 3 twin examples and the 6 ready examples, push what the model says (optional)

In [ ]:
OUT = f'results/R10/examples_{VERDICT}{"_fast" if FAST else ""}.json'
!python -m src.demo.app --run-examples {OUT}
# PUSHED is printed only if the examples ran, and the commit and push worked
!git checkout -q -B rathi/R10-video && git add {OUT} && git commit -q -m "[R10] Demo example results ({VERDICT} verdict)" && git push -q -f origin rathi/R10-video && echo PUSHED

## 4. Launch the demo (click the `https://….gradio.live` link). This cell keeps running while the demo is open; stop it when you are done.

In [ ]:
!python -m src.demo.app --share --evidence-source {EVIDENCE}